In [1]:
!pip install transformers torch nlpaug -q

from google.colab import drive
drive.mount('/content/drive')

import torch, torch.nn as nn, torch.nn.functional as F
import json, random, numpy as np
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import BertTokenizer, BertModel, get_linear_schedule_with_warmup
import nlpaug.augmenter.char as nac
import nlpaug.augmenter.word as naw
from sklearn.metrics import accuracy_score
import nltk
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)

device    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SAVE_DIR  = "/content/drive/MyDrive/NLP/bert-151"
MAX_LEN   = 64
OOS_ID    = 150
NUM_CLASS = 151

import os; os.makedirs(SAVE_DIR, exist_ok=True)
print(f"Device: {device}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.5/410.5 kB 9.1 MB/s eta 0:00:00
Mounted at /content/drive
Device: cuda


In [2]:
with open("/content/drive/MyDrive/NLP/oos-eval/data/data_full.json") as f:
    data = json.load(f)

# In-scope
train_texts  = [x[0] for x in data["train"]]
train_labels = [x[1] for x in data["train"]]
val_texts    = [x[0] for x in data["val"]]
val_labels   = [x[1] for x in data["val"]]
test_texts   = [x[0] for x in data["test"]]
test_labels  = [x[1] for x in data["test"]]

# OOS
oos_train_texts = [x[0] for x in data["oos_train"]]
oos_val_texts   = [x[0] for x in data["oos_val"]]
oos_test_texts  = [x[0] for x in data["oos_test"]]

# Label mapping — 150 in-scope + OOS as 150
in_scope    = sorted(set(train_labels))
label2id    = {l: i for i, l in enumerate(in_scope)}
label2id["oos"] = OOS_ID

# Combine in-scope + OOS
all_train_texts  = train_texts  + oos_train_texts
all_train_labels = [label2id[l] for l in train_labels] + [OOS_ID] * len(oos_train_texts)
all_val_texts    = val_texts    + oos_val_texts
all_val_labels   = [label2id[l] for l in val_labels]   + [OOS_ID] * len(oos_val_texts)
all_test_texts   = test_texts   + oos_test_texts
all_test_labels  = [label2id[l] for l in test_labels]  + [OOS_ID] * len(oos_test_texts)

# Separate test IDs for in-scope and OOS evaluation
inscope_test_ids = [label2id[l] for l in test_labels]
oos_test_ids     = [OOS_ID] * len(oos_test_texts)

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

print(f"Train: {len(all_train_texts)} | Val: {len(all_val_texts)} | Test: {len(all_test_texts)}")
print(f"OOS test samples: {len(oos_test_texts)}")
print(f"Total classes: {NUM_CLASS}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Train: 15100 | Val: 3100 | Test: 5500
OOS test samples: 1000
Total classes: 151


In [3]:
class IntentDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts   = texts
        self.labels  = labels
        self.tok     = tokenizer
        self.max_len = max_len

    def __len__(self): return len(self.labels)

    def __getitem__(self, idx):
        enc = self.tok(self.texts[idx], truncation=True,
                       padding="max_length", max_length=self.max_len,
                       return_tensors="pt")
        return {
            "input_ids":      enc["input_ids"].squeeze(),
            "attention_mask": enc["attention_mask"].squeeze(),
            "label":          torch.tensor(self.labels[idx])
        }

class BertIntent(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.bert       = BertModel.from_pretrained("bert-base-uncased")
        self.dropout    = nn.Dropout(0.1)
        self.classifier = nn.Linear(768, num_classes)
        self.prototypes = None

    def get_embedding(self, input_ids, attention_mask):
        out = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        return self.dropout(out.last_hidden_state[:, 0, :])

    def forward(self, input_ids, attention_mask, labels=None):
        emb    = self.get_embedding(input_ids, attention_mask)
        logits = self.classifier(emb)
        if labels is not None:
            loss = nn.CrossEntropyLoss()(logits, labels)
            return loss, logits
        return logits

def compute_metrics(model, texts, label_ids, batch_size=64, prototypes=None):
    """Returns accuracy and OOS recall."""
    model.eval()
    preds, labels_all = [], []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        enc   = tokenizer(batch, truncation=True, padding=True,
                          max_length=MAX_LEN, return_tensors="pt")
        with torch.no_grad():
            emb = model.get_embedding(enc["input_ids"].to(device),
                                      enc["attention_mask"].to(device))
            if prototypes is not None:
                dist   = torch.cdist(emb, prototypes)
                pred   = torch.argmin(dist, dim=-1).cpu().numpy()
            else:
                logits = model.classifier(emb)
                pred   = torch.argmax(logits, dim=-1).cpu().numpy()
        preds.extend(pred)
        labels_all.extend(label_ids[i:i+batch_size])

    preds      = np.array(preds)
    labels_all = np.array(labels_all)
    acc        = accuracy_score(labels_all, preds)

    # OOS recall — only on OOS samples
    oos_mask   = labels_all == OOS_ID
    oos_recall = (preds[oos_mask] == OOS_ID).mean() if oos_mask.any() else 0.0

    return acc, oos_recall

def print_results(model, label="", prototypes=None):
    """Print accuracy and OOS recall across all noise types."""
    noisy_data_map = {
        "original":      test_texts,
        "casing":        [t.upper() for t in test_texts],
        "keyboard":      add_keyboard_noise(test_texts),
        "spelling":      add_spelling_noise(test_texts),
        "synonyms":      add_synonym_noise(test_texts),
        "abbreviations": add_abbreviation_noise(test_texts),
    }
    print(f"\n{'─'*60}")
    print(f"Results: {label}")
    print(f"{'─'*60}")
    print(f"{'Noise':<20} {'In-scope Acc':>14} {'OOS Recall':>12}")
    print(f"{'─'*60}")
    for noise_type, texts in noisy_data_map.items():
        acc, oos_r = compute_metrics(model, texts, inscope_test_ids,
                                     prototypes=prototypes)
        print(f"{noise_type:<20} {acc*100:>13.2f}% {oos_r*100:>11.2f}%")

    # OOS recall on OOS test set
    _, oos_r = compute_metrics(model, oos_test_texts, oos_test_ids,
                               prototypes=prototypes)
    print(f"{'─'*60}")
    print(f"{'OOS test recall':<20} {'—':>14} {oos_r*100:>11.2f}%")

# Noise functions
kb_aug = nac.KeyboardAug(aug_char_p=0.15)
sp_aug = naw.SpellingAug(aug_p=0.2)
sy_aug = naw.SynonymAug(aug_p=0.2)

ABBREV_MAP = {
    "please": "pls", "you": "u", "your": "ur", "are": "r",
    "okay": "ok", "thanks": "thx", "tomorrow": "tmrw", "because": "bc",
    "with": "w/", "without": "w/o", "information": "info",
    "application": "app", "number": "num", "message": "msg",
    "account": "acct", "department": "dept", "appointment": "appt",
    "maximum": "max", "minimum": "min", "approximately": "approx",
    "password": "pwd", "transfer": "trnsfr", "tonight": "2nite",
    "want": "wnt", "where": "whr", "would": "wld",
}

def add_keyboard_noise(texts):
    out = []
    for t in texts:
        try: out.append(kb_aug.augment(t)[0])
        except: out.append(t)
    return out

def add_spelling_noise(texts):
    out = []
    for t in texts:
        try: out.append(sp_aug.augment(t)[0])
        except: out.append(t)
    return out

def add_synonym_noise(texts):
    out = []
    for t in texts:
        try: out.append(sy_aug.augment(t)[0])
        except: out.append(t)
    return out

def add_abbreviation_noise(texts):
    out = []
    for t in texts:
        words = t.split()
        out.append(" ".join(ABBREV_MAP.get(w.lower(), w) for w in words))
    return out

def generate_noisy(text):
    choice = random.choices(["keyboard", "spelling"], weights=[0.7, 0.3])[0]
    try:
        return kb_aug.augment(text)[0] if choice == "keyboard" else sp_aug.augment(text)[0]
    except:
        return text

print("All functions defined")

All functions defined


In [4]:
model = BertIntent(num_classes=NUM_CLASS).to(device)

train_dataset = IntentDataset(all_train_texts, all_train_labels, tokenizer, MAX_LEN)
val_dataset   = IntentDataset(all_val_texts,   all_val_labels,   tokenizer, MAX_LEN)
train_loader  = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader    = DataLoader(val_dataset,   batch_size=64)

EPOCHS    = 8
optimizer = AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
scheduler = get_linear_schedule_with_warmup(optimizer,
    num_warmup_steps=len(train_loader),
    num_training_steps=len(train_loader) * EPOCHS)

print("Starting clean BERT training (151 classes)...")
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for step, batch in enumerate(train_loader):
        ids    = batch["input_ids"].to(device)
        mask   = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)
        optimizer.zero_grad()
        loss, _ = model(ids, mask, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
        if step % 200 == 0:
            print(f"Epoch {epoch+1} Step {step}/{len(train_loader)} Loss: {loss.item():.4f}")

    # Val accuracy
    model.eval()
    val_preds, val_labels_all = [], []
    with torch.no_grad():
        for batch in val_loader:
            ids    = batch["input_ids"].to(device)
            mask   = batch["attention_mask"].to(device)
            logits = model(ids, mask)
            val_preds.extend(torch.argmax(logits, dim=-1).cpu().numpy())
            val_labels_all.extend(batch["label"].numpy())
    val_acc = accuracy_score(val_labels_all, val_preds)

    print(f"Epoch {epoch+1} done — Loss: {total_loss/len(train_loader):.4f} | Val Acc: {val_acc*100:.2f}%")
    torch.save(model.state_dict(), f"{SAVE_DIR}/bert_clean_epoch{epoch+1}.pt")
    print(f"Saved epoch {epoch+1}")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Starting clean BERT training (151 classes)...
Epoch 1 Step 0/472 Loss: 5.2470
Epoch 1 Step 200/472 Loss: 4.7759
Epoch 1 Step 400/472 Loss: 2.9606
Epoch 1 done — Loss: 4.1795 | Val Acc: 79.10%
Saved epoch 1
Epoch 2 Step 0/472 Loss: 2.0392
Epoch 2 Step 200/472 Loss: 0.7540
Epoch 2 Step 400/472 Loss: 0.6186
Epoch 2 done — Loss: 0.9012 | Val Acc: 92.35%
Saved epoch 2
Epoch 3 Step 0/472 Loss: 0.2649
Epoch 3 Step 200/472 Loss: 0.1114
Epoch 3 Step 400/472 Loss: 0.1447
Epoch 3 done — Loss: 0.2026 | Val Acc: 94.19%
Saved epoch 3
Epoch 4 Step 0/472 Loss: 0.1030
Epoch 4 Step 200/472 Loss: 0.0466
Epoch 4 Step 400/472 Loss: 0.1907
Epoch 4 done — Loss: 0.0772 | Val Acc: 94.97%
Saved epoch 4
Epoch 5 Step 0/472 Loss: 0.0292
Epoch 5 Step 200/472 Loss: 0.0205
Epoch 5 Step 400/472 Loss: 0.0172
Epoch 5 done — Loss: 0.0398 | Val Acc: 95.03%
Saved epoch 5
Epoch 6 Step 0/472 Loss: 0.0197
Epoch 6 Step 200/472 Loss: 0.0140
Epoch 6 Step 400/472 Loss: 0.0130
Epoch 6 done — Loss: 0.0251 | Val Acc: 94.97%
Saved ep

In [5]:
# Load best if needed:
model = BertIntent(num_classes=NUM_CLASS).to(device)
model.load_state_dict(torch.load(f"{SAVE_DIR}/bert_clean_epoch5.pt"))
model.eval()
print_results(model, label="BERT Clean (151 classes)")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Datele de ieșire de afișat au fost trunchiate la ultimele 5000 linii.
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]   


────────────────────────────────────────────────────────────
Results: BERT Clean (151 classes)
────────────────────────────────────────────────────────────
Noise                  In-scope Acc   OOS Recall
────────────────────────────────────────────────────────────
original                     96.00%        0.00%
casing                       96.00%        0.00%
keyboard                     44.24%        0.00%
spelling                     82.56%        0.00%
synonyms                     96.00%        0.00%
abbreviations                94.27%        0.00%
────────────────────────────────────────────────────────────
OOS test recall                   —       39.00%


In [6]:
# Load clean checkpoint if new session:
# model = BertIntent(num_classes=NUM_CLASS).to(device)
# model.load_state_dict(torch.load(f"{SAVE_DIR}/bert_clean_epoch5.pt"))

class ContrastiveDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts   = texts
        self.labels  = labels
        self.tok     = tokenizer
        self.max_len = max_len

    def __len__(self): return len(self.labels)

    def __getitem__(self, idx):
        def enc(t):
            return self.tok(t, truncation=True, padding="max_length",
                            max_length=self.max_len, return_tensors="pt")
        c = enc(self.texts[idx])
        n = enc(generate_noisy(self.texts[idx]))
        return {
            "clean_input_ids":      c["input_ids"].squeeze(),
            "clean_attention_mask": c["attention_mask"].squeeze(),
            "noisy_input_ids":      n["input_ids"].squeeze(),
            "noisy_attention_mask": n["attention_mask"].squeeze(),
            "label": torch.tensor(self.labels[idx])
        }

# Only use in-scope for contrastive pairs — OOS noise pairs don't make semantic sense
cont_dataset = ContrastiveDataset(train_texts,
                                   [label2id[l] for l in train_labels],
                                   tokenizer, MAX_LEN)
cont_loader  = DataLoader(cont_dataset, batch_size=16, shuffle=True)

CONT_EPOCHS = 3
optimizer   = AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
scheduler   = get_linear_schedule_with_warmup(optimizer,
    num_warmup_steps=len(cont_loader),
    num_training_steps=len(cont_loader) * CONT_EPOCHS)

print("Starting contrastive fine-tuning...")
for epoch in range(CONT_EPOCHS):
    model.train()
    total_loss = 0
    for step, batch in enumerate(cont_loader):
        c_ids  = batch["clean_input_ids"].to(device)
        c_mask = batch["clean_attention_mask"].to(device)
        n_ids  = batch["noisy_input_ids"].to(device)
        n_mask = batch["noisy_attention_mask"].to(device)
        labels = batch["label"].to(device)

        clean_emb = model.get_embedding(c_ids, c_mask)
        noisy_emb = model.get_embedding(n_ids, n_mask)
        logits    = model.classifier(clean_emb)

        ce_loss = nn.CrossEntropyLoss()(logits, labels)

        cn   = F.normalize(clean_emb, dim=-1)
        nn_  = F.normalize(noisy_emb, dim=-1)
        B    = cn.size(0)
        embs = torch.cat([cn, nn_], dim=0)
        sim  = torch.matmul(embs, embs.T) / 0.07
        mask = torch.eye(2*B, device=embs.device).bool()
        sim.masked_fill_(mask, float('-inf'))
        cl_labels = torch.cat([torch.arange(B, 2*B), torch.arange(B)]).to(embs.device)
        cl_loss   = nn.CrossEntropyLoss()(sim, cl_labels)

        loss = ce_loss + 0.3 * cl_loss
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()

        if step % 200 == 0:
            print(f"Epoch {epoch+1} Step {step}/{len(cont_loader)} "
                  f"Loss: {loss.item():.4f} CE: {ce_loss.item():.4f} CL: {cl_loss.item():.4f}")

    print(f"Epoch {epoch+1} done — Avg Loss: {total_loss/len(cont_loader):.4f}")
    torch.save(model.state_dict(), f"{SAVE_DIR}/bert_contrastive_epoch{epoch+1}.pt")
    print(f"Saved contrastive epoch {epoch+1}")

Starting contrastive fine-tuning...
Epoch 1 Step 0/938 Loss: 0.5538 CE: 0.0176 CL: 1.7873
Epoch 1 Step 200/938 Loss: 0.3182 CE: 0.0289 CL: 0.9645
Epoch 1 Step 400/938 Loss: 0.1023 CE: 0.0263 CL: 0.2533
Epoch 1 Step 600/938 Loss: 0.1046 CE: 0.0188 CL: 0.2858
Epoch 1 Step 800/938 Loss: 0.1929 CE: 0.0184 CL: 0.5815
Epoch 1 done — Avg Loss: 0.2193
Saved contrastive epoch 1
Epoch 2 Step 0/938 Loss: 0.0627 CE: 0.0159 CL: 0.1559
Epoch 2 Step 200/938 Loss: 0.1190 CE: 0.0161 CL: 0.3431
Epoch 2 Step 400/938 Loss: 0.1132 CE: 0.0133 CL: 0.3330
Epoch 2 Step 600/938 Loss: 0.1410 CE: 0.0179 CL: 0.4100
Epoch 2 Step 800/938 Loss: 0.0713 CE: 0.0269 CL: 0.1480
Epoch 2 done — Avg Loss: 0.0998
Saved contrastive epoch 2
Epoch 3 Step 0/938 Loss: 0.0494 CE: 0.0137 CL: 0.1192
Epoch 3 Step 200/938 Loss: 0.0829 CE: 0.0140 CL: 0.2297
Epoch 3 Step 400/938 Loss: 0.0989 CE: 0.0172 CL: 0.2721
Epoch 3 Step 600/938 Loss: 0.0495 CE: 0.0157 CL: 0.1125
Epoch 3 Step 800/938 Loss: 0.0710 CE: 0.0067 CL: 0.2144
Epoch 3 done —

In [7]:
print_results(model, label="BERT + Contrastive (151 classes)")

Datele de ieșire de afișat au fost trunchiate la ultimele 5000 linii.
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]   


────────────────────────────────────────────────────────────
Results: BERT + Contrastive (151 classes)
────────────────────────────────────────────────────────────
Noise                  In-scope Acc   OOS Recall
────────────────────────────────────────────────────────────
original                     96.73%        0.00%
casing                       96.73%        0.00%
keyboard                     74.31%        0.00%
spelling                     90.13%        0.00%
synonyms                     96.73%        0.00%
abbreviations                96.38%        0.00%
────────────────────────────────────────────────────────────
OOS test recall                   —       10.40%


In [8]:
# Load contrastive checkpoint if new session:
# model = BertIntent(num_classes=NUM_CLASS).to(device)
# model.load_state_dict(torch.load(f"{SAVE_DIR}/bert_contrastive_epoch3.pt"))

print("Building prototypes from training set...")
model.eval()
embs_per_class = {i: [] for i in range(NUM_CLASS)}

# Use in-scope training data for prototypes
for i, (text, label) in enumerate(zip(train_texts, [label2id[l] for l in train_labels])):
    enc = tokenizer(text, truncation=True, padding="max_length",
                    max_length=MAX_LEN, return_tensors="pt")
    with torch.no_grad():
        emb = model.get_embedding(enc["input_ids"].to(device),
                                   enc["attention_mask"].to(device))
    embs_per_class[label].append(emb.squeeze().cpu())
    if i % 2000 == 0:
        print(f"  {i}/{len(train_texts)}")

# Add OOS prototype from OOS training examples
oos_embs = []
for text in oos_train_texts:
    enc = tokenizer(text, truncation=True, padding="max_length",
                    max_length=MAX_LEN, return_tensors="pt")
    with torch.no_grad():
        emb = model.get_embedding(enc["input_ids"].to(device),
                                   enc["attention_mask"].to(device))
    oos_embs.append(emb.squeeze().cpu())

embs_per_class[OOS_ID] = oos_embs

prototypes = torch.stack([
    torch.stack(embs_per_class[i]).mean(dim=0)
    for i in range(NUM_CLASS)
]).to(device)

torch.save(prototypes, f"{SAVE_DIR}/prototypes.pt")
print("Prototypes built and saved")

print_results(model, label="BERT + Contrastive + Prototypical (151 classes)",
              prototypes=prototypes)

Building prototypes from training set...
  0/15000
  2000/15000
  4000/15000
  6000/15000
  8000/15000
  10000/15000
  12000/15000
  14000/15000
Prototypes built and saved


Datele de ieșire de afișat au fost trunchiate la ultimele 5000 linii.
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]   


────────────────────────────────────────────────────────────
Results: BERT + Contrastive + Prototypical (151 classes)
────────────────────────────────────────────────────────────
Noise                  In-scope Acc   OOS Recall
────────────────────────────────────────────────────────────
original                     96.44%        0.00%
casing                       96.44%        0.00%
keyboard                     74.42%        0.00%
spelling                     89.56%        0.00%
synonyms                     96.44%        0.00%
abbreviations                96.09%        0.00%
────────────────────────────────────────────────────────────
OOS test recall                   —       31.50%
